# Uncertainty intervals (Figure A1)

Empirical coverage of each method's 95% prediction interval by forecast horizon,
over jump-off years 1985-2013 (Appendix C of the paper). De Beer and freeze rates
are left out because Bohk-Ewald, Li & Myrskyla (2018) do not report intervals for
them.

## Load intervals and observed rates

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Arial throughout, to match the other figures in the paper
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "mathtext.fontset": "stixsans",
    "pdf.fonttype": 42,   # embed TrueType fonts
    "ps.fonttype": 42,
})

DATA_DIR = '../../data'
FIG_DIR = '../../figures'

# label -> interval file, observed file, and styling (same colours and markers as
# the other figures in the paper)
UNC = {
    'Lee':                dict(unc='lee2_forecasts_cohort_uncertainty',
                               obs='lee2_obs_cohort',            color='#d62728', marker='o'),
    'Myrskyla et al.':    dict(unc='myrskyla_forecasts_cohort_uncertainty',
                               obs='myrskyla_obs_cohort',        color='#9467bd', marker='P'),
    'Schmertmann et al.': dict(unc='schmertmann2013_forecasts_cohort_uncertainty',
                               obs='schmertmann2013_obs_cohort', color='#7f7f7f', marker='D'),
    'Sevcikova et al.':   dict(unc='sevcikova_forecasts_cohort_uncertainty',
                               obs='sevcikova_obs_cohort',       color='#e377c2', marker='h'),
    'CCF-Net':            dict(unc='dl_forecasts_CCF_Net_u110_l4_d0.0_w40_uncertainty',
                               obs='dl_obs_cohort_CCF_Net_u110_l4_d0.0_w40',
                               color='#8c564b', marker='*'),
}
NOMINAL = 0.95

# Attach the observed rate (Obs) to each forecast cell. Observed rates don't
# depend on the jump-off year, so they are matched on (Country, Year, Age), where
# Year is the birth cohort.
OBS_KEY = ['Country', 'Year', 'Age']
for m in UNC.values():
    unc = pd.read_csv(f"{DATA_DIR}/{m['unc']}.csv")
    obs = (pd.read_csv(f"{DATA_DIR}/{m['obs']}.csv")
           .drop_duplicates(OBS_KEY)[OBS_KEY + ['Rate']].rename(columns={'Rate': 'Obs'}))
    m['df'] = unc.merge(obs, on=OBS_KEY, how='inner')

## Coverage by forecast horizon

In [ ]:
# Coverage = share of forecast cells whose observed rate falls inside the 95%
# interval. Only forecast cells count (horizon >= 1): the benchmark files also
# include the observed history, whose zero-width intervals (Lee, Myrskyla) would
# count as trivially covered.
MATCH_COLS = ['Country', 'Year', 'Age', 'JumpOffYear']
MIN_CELLS = 200     # horizons with fewer cells than this are not plotted

for m in UNC.values():
    d = m['df']
    d['Horizon'] = d['Year'] + d['Age'] - d['JumpOffYear']   # calendar years past JOY
    d = d[d['Horizon'] >= 1].copy()
    d['Covered'] = d['Obs'].between(d['Lower95'], d['Upper95'])
    m['df'] = d

# Compare all methods on the same cells: only those every method forecasts.
common_keys = None
for m in UNC.values():
    k = m['df'][MATCH_COLS]
    common_keys = k if common_keys is None else common_keys.merge(k, on=MATCH_COLS)
for m in UNC.values():
    m['df'] = m['df'].merge(common_keys, on=MATCH_COLS)
print(f"Common grid: {len(common_keys):,} forecast cells, "
      f"{common_keys['Country'].nunique()} countries")

by_h = {}
for label, m in UNC.items():
    t = m['df'].groupby('Horizon').agg(Coverage=('Covered', 'mean'), N=('Covered', 'size'))
    by_h[label] = t[t['N'] >= MIN_CELLS]

## Figure A1

In [ ]:
# Figure A1: empirical coverage of the 95% interval by forecast horizon
fig, ax = plt.subplots(figsize=(8, 5))
ax.axhline(NOMINAL, color='black', lw=1, ls='--', zorder=1)
ax.text(25, NOMINAL + 0.012, 'Nominal 95%', fontsize=10, color='0.25', va='bottom', ha='right')
for label, m in UNC.items():
    t = by_h[label]
    ax.plot(t.index, t['Coverage'], color=m['color'], marker=m['marker'],
            lw=2, ms=6, markevery=2, label=label)
ax.set_ylim(0, 1.08)
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_xlabel('Forecast horizon (years past jump-off)', fontsize=13)
ax.set_ylabel('Coverage of the 95% interval', fontsize=13)
ax.tick_params(labelsize=11)
ax.grid(axis='y', color='0.9', lw=0.8)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False, fontsize=11, ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.14))
plt.tight_layout()

os.makedirs(FIG_DIR, exist_ok=True)
plt.savefig(f'{FIG_DIR}/uncertainty_coverage_by_horizon.pdf', format='pdf', bbox_inches='tight')
plt.show()